Fake News Classifier using Bidirectional lstm

In [14]:
import pandas as pd

In [15]:
data = pd.read_csv("data.csv")

In [16]:
data.head()

,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1


In [17]:
data.drop(columns="date", axis=1, inplace=True)

In [18]:
data.head()

,title,text,subject,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,1


In [19]:
data.tail()

,title,text,subject,label
44893,UNREAL! CBS’S TED KOPPEL Tells Sean Hannity He...,,politics,0
44894,PM May seeks to ease Japan's Brexit fears duri...,LONDON/TOKYO (Reuters) - British Prime Ministe...,worldnews,1
44895,Merkel: Difficult German coalition talks can r...,BERLIN (Reuters) - Chancellor Angela Merkel sa...,worldnews,1
44896,Trump Stole An Idea From North Korean Propaga...,Jesus f*cking Christ our President* is a moron...,News,0
44897,BREAKING: HILLARY CLINTON’S STATE DEPARTMENT G...,IF SHE S NOT TOAST NOW THEN WE RE IN BIGGER TR...,politics,0


In [20]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44898 non-null  object
 1   text     44898 non-null  object
 2   subject  44898 non-null  object
 3   label    44898 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 1.4+ MB


In [21]:
data.isnull().sum()

title      0
text       0
subject    0
label      0
dtype: int64

In [22]:
data = data.dropna()

In [23]:
data.shape

(44898, 4)

In [24]:
X = data.drop('label', axis=1)

In [25]:
Y = data['label']

In [26]:
import tensorflow as tf

In [27]:
tf.__version__

'2.21.0'

In [28]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM , Bidirectional

In [29]:
### Vocab size

voc_size = 10000

One Hot Representation

In [30]:
messages = X.copy()

In [31]:
messages['title'][1]

'Trump drops Steve Bannon from National Security Council'

In [32]:
messages.reset_index(inplace=True)

In [33]:
import nltk
import re
from nltk.corpus import stopwords

In [34]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [35]:
### Dataset Preprocessing
from nltk.stem.porter import PorterStemmer ##stemming purpose
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
    review = review.lower()
    review = review.split()
    
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

In [36]:
corpus

['ben stein call th circuit court commit coup tat constitut',
 'trump drop steve bannon nation secur council',
 'puerto rico expect u lift jone act ship restrict',
 'oop trump accident confirm leak isra intellig russia video',
 'donald trump head scotland reopen golf resort',
 'paul ryan respond dem sit gun control disgust way video',
 'awesom diamond silk rip press believ video',
 'stand cheer ukip parti leader slam germani franc eu invas phoni refuge video',
 'north korea show sign seriou talk u offici',
 'trump signal willing rais u minimum wage',
 'new jersey christi mull run lead republican parti report',
 'hillari clinton spot dine alon',
 'franc germani want iran revers ballist missil program',
 'aid eu commiss head tweet pictur white smoke brexit meet may',
 'trump issu warn man armi could isi video',
 'u give lao extra million help clear unexplod ordnanc',
 'judg declar babi name illeg prevent emot harm',
 'paul ryan take monument humili photo constitu expertli troll imag',
 '

In [37]:
one_hot_repr = [one_hot(words, voc_size) for words in corpus]
one_hot_repr

[[2833, 9966, 9547, 2213, 8509, 5687, 1454, 2979, 4162, 2979],
 [7285, 6873, 9436, 3186, 9169, 49, 9182],
 [6961, 9636, 8126, 8331, 7710, 8574, 6852, 8171, 7452],
 [9850, 7285, 3044, 1766, 1783, 4995, 3629, 2905, 1613],
 [5555, 7285, 5981, 9597, 6407, 7549, 3128],
 [783, 2307, 7630, 5822, 1450, 5666, 6694, 960, 3616, 1613],
 [2090, 6434, 9437, 6776, 2218, 3178, 1613],
 [8684, 8818, 1067, 998, 6989, 3923, 7560, 3331, 9081, 6667, 8004, 6199, 1613],
 [1819, 7622, 8152, 7179, 131, 7166, 8331, 5453],
 [7285, 5223, 1435, 349, 8331, 3501, 3386],
 [5989, 3876, 4788, 7771, 56, 6088, 5320, 998, 1813],
 [2470, 2032, 5434, 7278, 7032],
 [3331, 7560, 6670, 3995, 4682, 7382, 4601, 2467],
 [4156, 9081, 7545, 5981, 79, 711, 9563, 1708, 9168, 2825, 8624],
 [7285, 8507, 4626, 3771, 8360, 2100, 1936, 1613],
 [8331, 1998, 8307, 428, 2466, 1157, 4315, 3398, 4745],
 [4264, 8796, 6442, 2356, 4883, 756, 3327, 1579],
 [783, 2307, 866, 4031, 4212, 2504, 1498, 5085, 3290, 6419],
 [5320, 7278, 7285, 3157, 2420, 7

Embedding Representation

In [38]:
sent_length = 20
embedded_docs = pad_sequences(one_hot_repr, padding='pre' , maxlen=sent_length)
embedded_docs

array([[   0,    0,    0, ..., 2979, 4162, 2979],
       [   0,    0,    0, ..., 9169,   49, 9182],
       [   0,    0,    0, ..., 6852, 8171, 7452],
       ...,
       [   0,    0,    0, ..., 7166, 2328, 6181],
       [   0,    0,    0, ..., 6078, 3352, 1508],
       [   0,    0,    0, ..., 3578, 6146, 8900]],
      shape=(44898, 20), dtype=int32)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

model = Sequential([
    Input(shape=(sent_length,)),
    Embedding(voc_size, 40),
    Bidirectional(LSTM(100,
        dropout=0.3,
        recurrent_dropout=0.3)),  
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 40)         │       400,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 200)            │       112,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           201 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 513,001 (1.96 MB)

 Trainable params: 513,001 (1.96 MB)

 Non-trainable params: 0 (0.00 B)

In [40]:
len(embedded_docs), Y.shape

(44898, (44898,))

In [41]:
import numpy as np
X_final = np.array(embedded_docs)
Y_final = np.array(Y)

In [42]:
X_final.shape, Y_final.shape

((44898, 20), (44898,))

In [43]:
from sklearn.model_selection import train_test_split

X_train , X_test , Y_train , Y_test = train_test_split(X_final, Y_final, test_size=0.2, random_state=42)

In [44]:
## Model traning 
model.fit(X_train, Y_train, validation_data=(X_test, Y_test), epochs=10, batch_size=62)

Epoch 1/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 57s 60ms/step - accuracy: 0.9089 - loss: 0.2148 - val_accuracy: 0.9516 - val_loss: 0.1301
Epoch 2/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 34s 58ms/step - accuracy: 0.9572 - loss: 0.1158 - val_accuracy: 0.9509 - val_loss: 0.1252
Epoch 3/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 33s 57ms/step - accuracy: 0.9672 - loss: 0.0904 - val_accuracy: 0.9547 - val_loss: 0.1206
Epoch 4/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 34s 58ms/step - accuracy: 0.9748 - loss: 0.0697 - val_accuracy: 0.9533 - val_loss: 0.1231
Epoch 5/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 33s 57ms/step - accuracy: 0.9779 - loss: 0.0591 - val_accuracy: 0.9529 - val_loss: 0.1469
Epoch 6/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 33s 57ms/step - accuracy: 0.9832 - loss: 0.0488 - val_accuracy: 0.9527 - val_loss: 0.1456
Epoch 7/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 34s 58ms/step - accuracy: 0.9858 - loss: 0.0390 - val_accuracy: 0.9507 - val_loss: 0.1721
Epoch 8/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 33s 57ms/step - accuracy: 0.9880 - loss: 0.0341 - 

In [45]:
y_pred = model.predict(X_test)


281/281 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step


In [46]:
y_pred = np.where(y_pred > 0.5,1,0)

In [47]:
from sklearn.metrics import confusion_matrix


In [48]:
confusion_matrix(Y_test, y_pred)



array([[4488,  222],
       [ 237, 4033]])

In [49]:
from sklearn.metrics import accuracy_score
accuracy_score(Y_test, y_pred)

0.9488864142538975

In [50]:
from sklearn.metrics import classification_report
print(classification_report(Y_test , y_pred))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95      4710
           1       0.95      0.94      0.95      4270

    accuracy                           0.95      8980
   macro avg       0.95      0.95      0.95      8980
weighted avg       0.95      0.95      0.95      8980

